# Intro To Handling Dates In Pandas
Properly handling dates in Pandas can be very useful.  For example, let's say you read in a csv of temperature data with dates and you'd like to find the monthly mean temperature.  If the date column is already recognized, by pandas, as a date type, then extracting the month from the column will be very easy.  If the date column is, instead, read in as a string type, then it will be more difficult to extract the month.

## Notebook Outline
* [Using the parse dates .read_csv() to automatically read in dateformats](#parsedates)
* [Using .to_datetime() to convert a column to a pandas datetime format](#todatetime)
* [Using .dt on datetime columns](#dtattribute)
* [Introduction the .min(), .max() and .sum() methods](#minmaxsum)

In [ ]:
import pandas as pd

<a name=parsedates></a>
# Using the parse_dates argument in .read_csv() to automatically read in date formats
When reading in a file, we can specify which columns, or which combinations of columns, we would like read in as a datetime type. For this example, I am going to introduce a new data file 'LaborSheetData.csv'. This file contains real data from a very popular fast food store. Every hour, the shift manager must enter some key data in this file, like drive through times and sales for the past hour. This will be a good dataset for us to explore in some of our lectures.

First we will load the data. Remember that you will need to change the path to point to where you place the file on your computer, after you download it.

In [ ]:
filepath = ('../Data/LaborSheetData.csv')
laborSheetData = pd.read_csv(filepath)

#### Use the .head() method to get a look at the data, and the .info method to see the data types

In [ ]:
laborSheetData.head(2)

In [ ]:
laborSheetData.info()

#### Note that data types of the Date and TimeStamp columns
The data types are object, which means they were read in as strings. Let's double check that by grabbing a value from the data column using the .loc() method and finding its type with the type function.

In [ ]:
type(laborSheetData.loc[0, 'Date'])

In [ ]:
laborSheetData.loc[:, ('Date','Manager','KVS Total')].groupby(by = 'Date').mean()

#### Now, we will use the parse_dates argument to automatically read the date columns in as date types.
All we need to do is pass a list of the columns we would like pandas to try and automatically decipher as date or time objects to the parse_dates argument.  If you look above at the output from .info, you will see that the date or datetime columns are columns 2 and 13 (remember that counting starts at 0).

Note that this will cause the read_csv() method to take a little bit longer to complete.

In [ ]:
laborSheetData = pd.read_csv(filepath, parse_dates=[2, 13])

In [ ]:
laborSheetData.head(2)

#### Now use .info() to check our datatypes.
Note that the 'Date' and 'TimeStamp' columns are now datetime types!

In [ ]:
laborSheetData.info()

#### Now let's use the parse_dates argument to _combine_ two (or more) columns into one datetime. 
For this, I need to give you some information about this data. The 'TimeStamp' column let's us know when the data was entered, but the 'Date' + 'Hour' column let us know what hour the data is for.  So it would be great if we could combine the Date and Hour columns into one datetime column!

We can do this using the parse_dates argument. All we need to do is include a list, of the columns we want to combine, as one of the items in the list that we we pass to the parse_dates argument.  

Note that pandas will create a _new_ column called 'Date_Hour' that combines the 'Date' and 'Hour' columns two columns.

In [ ]:
laborSheetData = pd.read_csv(filepath, parse_dates=[[2, 3], 13])

In [ ]:
laborSheetData.head(2)

In [ ]:
laborSheetData.info()

## Let's see another example using our weather data.
Firs we will load the weather data and use .head() to look at the data.  Note that the first four columns are Year, Month, Day, and Hour.

In [ ]:
filepath = ('../Data/724080-13739-2001')
headers = ['Year', 'Month', 'Day', 'Hour', 'Air Temp', 'Dew Point Temp', 'Sea Level Pressure',
           'Wind Direction', 'Wind Speed Rate',
           'Sky Condition Total Coverage Code',
           'Liquid Precipitation Depth Dimension - 1Hr Duration',
           'Liquid Precipitation Depth Dimension - Six Hour Duration']
weatherData = pd.read_csv(filepath, delim_whitespace=True,
                          names=headers)

In [ ]:
weatherData.head(1)

#### Now, use parse_dates to reload the data and combine the first four columns into one datetime column

In [ ]:
weatherData = pd.read_csv(filepath, delim_whitespace=True,
                          names=headers, parse_dates=[[0, 1, 2, 3]])

In [ ]:
weatherData.head(2)

In [ ]:
weatherData.info()

<a name=todatetime></a>
# Using pandas.to_datetime() to convert a column of strings to a columns of DateTime objects
Sometimes, we need to convert a column after we read in the data. Maybe we have created the column during data processing. We can do this with the to_datetime() function in the pandas package.

I am going to read in the labor sheet data again, without using parse_dates.

In [ ]:
filepath = ('../Data/LaborSheetData.csv')
laborSheetData = pd.read_csv(filepath)
laborSheetData.head()

In [ ]:
laborSheetData.info()

#### Use to_datetime to convert the 'TimeStamp' column to a column of datetime objects.
Note that to_datetime can not combine multiple columns (except under a few specific circumstances) so we can not use it to combine the 'Date' and 'Hour' columns.

In [ ]:
laborSheetData.loc[:, 'TimeStamp'] = pd.to_datetime(laborSheetData['TimeStamp'])

In [ ]:
laborSheetData.loc[:, 'TimeStamp'].groupby(by = '')

In [ ]:
laborSheetData.info()

<a name=dtattribute></a>
# Using .dt to access and operate on the datetime objects in a column
I am sure you are wondering why we went to the trouble to convert the columns to the datetime data type. Well it let's us manipulate the datetimes much more easily. Let's see some basics below:

First I read the data back in, correctly use the parse_dates argument.

In [ ]:
laborSheetData = pd.read_csv(filepath, parse_dates=[[2, 3], 13])
laborSheetData.head(2)

#### Use .dt to access the datetime properties, and then use .month to get the month of each value in the column

In [ ]:
laborSheetData['Date_Hour'].dt.month.head(2)

#### Use .dt and .year to get the year of each columns

In [ ]:
laborSheetData['Date_Hour'].dt.year.head(2)

#### Use .dt and .day to get the day of each columns

In [ ]:
laborSheetData['Date_Hour'].dt.day.head(2)

#### Use .dt and .hour to get the hour of each columns

In [ ]:
laborSheetData['Date_Hour'].dt.hour.head(2)

#### Use .dt and .hour and .value_counts() to get the row counts for each hour; do some hours get recorded less than others?

In [ ]:
laborSheetData['Date_Hour'].dt.hour.value_counts()

#### Introducing .sort_index().  The .sort_index() method will sort the index of a dataframe (while reordering the rows appropriately)

Note that the index of the value_counts() results is made up of the values that you are counting.

In [ ]:
laborSheetData['Date_Hour'].dt.hour.value_counts().sort_index()

<a name=minmaxsum></a>
## Introducing the .min(), .max(), and .sum() methods.
You can use these methods on any column, or dataframe to get the min, max, and sum of all numerical columns. You can also use the .min() and .max() methods to get the min and max of datetime columns

#### Use .min() and .max() to find the earliest and latest year in the data.

In [ ]:
print(laborSheetData['Date_Hour'].min())
print(laborSheetData['Date_Hour'].max())

#### Combine the use of .loc, .dt, and .sum() to find the total sales In August across all stores.

In [ ]:
laborSheetData.loc[laborSheetData['Date_Hour'].dt.month == 8, 'Sales'].sum()

#### Selecting rows greater than (or less than) a date.
Getting rows greater than, or less than (or some combination of logic tests) is fairly easy.  You use your normal logic tests: >, <, ==, etc...  and pass the value you'd like to test against as a string.  See the example below, note how we are able to use the string '2017-08-01' to get all rows with a value in the 'TimeStamp' column greater than '2017-08-01'. Easy!

In [ ]:
laborSheetData.loc[laborSheetData['TimeStamp'] > '2017-08-01', :].head()

# Changing the time zone of a datetime column
Pandas includes some easy functionality to convert the timezone of a datetime column.  The first thing we need to do is 'localize' the column - this means defining what timezone the original data is in.  They data we have been using happens to be from fast food location on the west coast, so we will localize the timestamp column to the 'US/Pacific' timezone by using the tz_localize() method.

In [ ]:
laborSheetData['TimeStamp'].dt.tz_localize('US/Pacific').head()

In [ ]:
laborSheetData.loc[:, 'TimeStamp'] = laborSheetData['TimeStamp'].dt.tz_localize('US/Pacific')

In [ ]:
laborSheetData.head()

In [ ]:
laborSheetData.loc[:, 'TimeStamp_UTC'] = laborSheetData['TimeStamp'].dt.tz_convert('UTC')

In [ ]:
laborSheetData.head()

# Using .date_range() to create a DateTime index

We can easily create a DateTime index by using the .date_range() method. We just need to pass a start date, and end date (or a number of periods) and a frequency (the amount of time between each value in the series of values).  Possible values for freq are:
* s (for seconds)
* min (for minutes)
* H (for hours)
* D (for days)
* A (for annual, ending on the end of a year)
* AS (for annual, ending on the start of a year)
* You can also do multiples, i.e. 3H for a step of 3 hours.
* You can also use something called a DateOffset object for more custom steps, but that is beyond the scope of this course.

Note that, once you create an index, you can use it as a row index or column in a dataframe.

The docs for this method are here: <https://pandas.pydata.org/pandas-docs/stable/generated/pandas.date_range.html>

Let's look at some examples:

#### Create a DatetimeIndex with a start date of 2001-01-01, an end date of 2002-01-01 and a step of 3 hours.

In [ ]:
pd.date_range(start='2001-01-01', end='2002-01-01', freq='3H')

#### Create a DatetimeIndex with a start date of 2001-01-01 and a  frequency of 1 day that continues for 100 periods.

In [ ]:
pd.date_range(start='2001-01-01', periods=100, freq='D')

#### Create a DatetimeIndex with a start date of 1970-01-01, that continues for 10 periods, and has an annual frequency - on the first day of each year.

In [ ]:
pd.date_range(start='1970-01-01', periods=10, freq='AS')

#### Create a DatetimeIndex with a start date of 1970-01-01, that continues for 10 periods, and has an annual frequency - on the last day of each year.

In [ ]:
pd.date_range(start='1969-12-31', periods=10, freq='A')

#### Create a DatetimeIndex with a start date of 2001-01-01 and a step size of 1 day that continues for 100 steps.

In [ ]:
pd.date_range(start='1969-12-31', periods=10, freq='A-AUG')